### Import modules and data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import numpy as np
import random

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.modeling.sarima_model import SARIMAModel
from backend.modeling.prophet_model import ProphetModel
from backend.modeling.xgboost_model import XGBoostModel
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
water_engineered_path = PATHS['engineered_data'] / 'water_engineered.parquet'
df = pd.read_parquet(water_engineered_path)
df.head()

### Training the different models to stack them:
- Prophet and Sarima are trained first
- Some of that data is used as training data for the XGBoost model, and XGBoost predicts the rest too
- From the dates that has predictions from all three models, a linear regression model is trained in the first years, and predicts the rest
- The predictions from all models are then combined using a full weighted average approach, where the weights are determined based on the performance of each model

In [ ]:
def prepare_data(df, reservoir_id):
    df_res = df[df['id'] == reservoir_id].copy()
    df_res = df_res.sort_values('date')

    capacity = df[df['id'] == reservoir_id]['capacity'].values[0]

    # Set date as index
    series = df_res.set_index('date')['storage']
    length = len(series)
    years = min(13, length // 52) - 1 # At most we get 10 years of data

    # Train/test split: last 52 weeks as test
    train_series = series.iloc[-(52*(years+1)):-52] # 'years' years

    train_stacking_index = train_series[-(52*(years-4)):].index
    # Create a dataframe with train_index as index
    train_stacking = pd.DataFrame(index=train_stacking_index)
    train_stacking['storage'] = train_series[-(52*(years-4)):] # 'years'-4 years, as the first 4 are for training sarima and prophet
    train_stacking['sarima_prediction'] = np.nan
    train_stacking['prophet_prediction'] = np.nan

    test_stacking = pd.DataFrame(index=series.iloc[-52:].index)
    test_stacking['storage'] = series.iloc[-52:] # 1 year
    test_stacking['sarima_prediction'] = np.nan
    test_stacking['prophet_prediction'] = np.nan

    return train_series, train_stacking, test_stacking, capacity, years

In [ ]:
def cv_sarima(train_sarima_prophet, train, test, years, capacity):
    for i in range(years-4):
        # print(f"Starting SARIMA model training for iteration {i}")
        sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
        sarima_model.fit(train_sarima_prophet.iloc[(52*i):(52*(i+4))])
        # print(f"\n Selecting from {train_series.index[-(52*(i+1)):-52*(i+4)]}")
        # print(f"\n train_series size: {train_series.shape}")
        sarima_test_pred = sarima_model.predict(steps=52)
        mask = sarima_test_pred > capacity
        sarima_test_pred[mask] = capacity
        train.iloc[(52*i):(52*(i+1)), train.columns.get_loc('sarima_prediction')] = sarima_test_pred

    # Predicting SARIMA for test data
    sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
    sarima_model.fit(train_sarima_prophet[-(52*4):])
    sarima_test_pred = sarima_model.predict(steps=52)
    mask = sarima_test_pred > capacity
    sarima_test_pred[mask] = capacity
    sarima_test_pred.index = test.index

    test['sarima_prediction'] = sarima_test_pred

    return train, test


In [ ]:
def cv_prophet(train_sarima_prophet, train, test, years, capacity):
    for i in range(years-4):
        # print(f"Starting Prophet model training for iteration {i}")
        prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
        prophet_model.fit(train_sarima_prophet.iloc[(52*i):(52*(i+4))])
        # print(f"\n Selecting from {train_series.index[(52*i):(52*(i+4))]}")
        # print(f"\n train_series size: {train_series.shape}")
        prophet_test_pred = prophet_model.predict(steps=52)
        # print(f"\n Shape of Prophet predictions for iteration {i}: {prophet_test_pred.shape}")
        mask = prophet_test_pred > capacity
        prophet_test_pred[mask] = capacity
        train.iloc[(52*i):(52*(i+1)), train.columns.get_loc('prophet_prediction')] = prophet_test_pred

    # Predicting prophet for test data
    prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
    prophet_model.fit(train_sarima_prophet[-(52*4):])
    # print(f"\n Selecting from {train_series.index[-(52*4):]}")
    prophet_test_pred = prophet_model.predict(steps=52)
    mask = prophet_test_pred > capacity
    prophet_test_pred[mask] = capacity
    prophet_test_pred.index = test.index

    test['prophet_prediction'] = prophet_test_pred
    # print(f"\nFinal train stacking DataFrame shape: {train_stacking.shape}")
    # test_stacking.info()

    return train, test


In [ ]:
def cv_xgboost(train, test, train_series_xgboost, years_training, years_training_xgboost, capacity):
    def predict_next_year(model, X, years_training_xgboost):
        predictions = []
        for week in range(1, 53):
            X_step = X.copy()[-(52*years_training_xgboost):]
            next_year_predictions = pd.concat([X_step, test], axis=0)
            X_step['sarima_prediction'] = next_year_predictions['sarima_prediction'].shift(-week).iloc[:52*years_training_xgboost]
            X_step['prophet_prediction'] = next_year_predictions['prophet_prediction'].shift(-week).iloc[:52*years_training_xgboost]
            storage_one_year_ago = X_step['storage'][-52]
            difference = storage_one_year_ago - X_step['storage'].iloc[-1]
            last_row = X_step.iloc[-1]
            X_step['next_storage_value'] = X_step['storage'].shift(-week)
            X_step = X_step.iloc[:-52]

            
            last_row['storage'] = last_row['storage'] + difference
            last_row['sarima_prediction'] = last_row['sarima_prediction'] + difference
            last_row['prophet_prediction'] = last_row['prophet_prediction'] + difference
            
            
            # print(f"\n X_step from step {week}: \n {X_step}")
            model.fit(X_step)
            
            # print(f"\n Last row for week {week}: \n {last_row}")
            # Generate forecasts for 52 steps
            prediction = model.predict(last_row.to_frame().T)
            predictions.append(prediction[0]-difference)
            # predictions.append(prediction[0])

        return pd.Series(predictions, dtype=float)
    
    # Create and train XGBoost model
    model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)

    for i in range(years_training):
        xgboost_test_pred = predict_next_year(model, train_series_xgboost.iloc[i*52:(i+years_training_xgboost)*52], years_training_xgboost)
        mask = xgboost_test_pred > capacity
        xgboost_test_pred[mask] = capacity
        train.iloc[(52*i):(52*(i+1)), train.columns.get_loc('xgboost_prediction')] = xgboost_test_pred.values.astype('float64')

    xgboost_test_pred = predict_next_year(model, train_series_xgboost.iloc[-(52*years_training_xgboost):], years_training_xgboost)
    mask = xgboost_test_pred > capacity
    xgboost_test_pred[mask] = capacity
    xgboost_test_pred.index = test.index

    test['xgboost_prediction'] = xgboost_test_pred.astype('float64')
    train['xgboost_prediction'] = train['xgboost_prediction'].astype(float)

    return train, test

#### Function to plot a reservoir

In [ ]:
def plot_one_reservoir(train, data_for_metrics, reservoir_id):
# Create comprehensive plot
    plt.figure(figsize=(15, 8))

    # Plot training data and predictions
    plt.plot(train.index, train['storage'], label='Train Actual', color='blue', linewidth=2)
    plt.plot(train.index, train['sarima_prediction'], label='SARIMA Train', color='red', linestyle='--', alpha=0.7)
    plt.plot(train.index, train['prophet_prediction'], label='Prophet Train', color='green', linestyle='--', alpha=0.7)
    plt.plot(train.index, train['xgboost_prediction'], label='XGBoost Train', color='purple', linestyle='--', alpha=0.7)

    # Plot test data and predictions
    plt.plot(data_for_metrics.index, data_for_metrics['storage'], label='Test Actual', color='black', linewidth=3)
    plt.plot(data_for_metrics.index, data_for_metrics['sarima_prediction'], label='SARIMA Test', color='red', alpha=0.8)
    plt.plot(data_for_metrics.index, data_for_metrics['prophet_prediction'], label='Prophet Test', color='green', alpha=0.8)
    plt.plot(data_for_metrics.index, data_for_metrics['xgboost_prediction'], label='XGBoost Test', color='purple', alpha=0.8)
    plt.plot(data_for_metrics.index, data_for_metrics['LR_prediction'], label='Stacking Test', color='orange', linewidth=2)
    plt.plot(data_for_metrics.index, data_for_metrics['weighted_average'], label='Weighted Average Test', color='cyan', linestyle='--', alpha=0.8)

    # Add train/test split line
    plt.axvline(data_for_metrics.index[0], color='gray', linestyle='--', alpha=0.8, label='Train/Test Split')

    # Formatting
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f'Stacking Ensemble Forecast for Reservoir {reservoir_id}')
    plt.xlabel('Date')
    plt.ylabel('Storage')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


#### Function to calculate the performance metrics for a single reservoir with all the models

In [ ]:
def calculate_reservoir_metrics(data):
    y_true = data['storage']
    metrics = {
        'prophet': {
            'mse': mean_squared_error(y_true, data['prophet_prediction']),
            'mae': mean_absolute_error(y_true, data['prophet_prediction']),
            'r2': r2_score(y_true, data['prophet_prediction'])
        },
        'sarima': {
            'mse': mean_squared_error(y_true, data['sarima_prediction']),
            'mae': mean_absolute_error(y_true, data['sarima_prediction']),
            'r2': r2_score(y_true, data['sarima_prediction'])
        },
        'xgboost': {
            'mse': mean_squared_error(y_true, data['xgboost_prediction']),
            'mae': mean_absolute_error(y_true, data['xgboost_prediction']),
            'r2': r2_score(y_true, data['xgboost_prediction'])
        },
        'linear_regression': {
            'mse': mean_squared_error(y_true, data['LR_prediction']),
            'mae': mean_absolute_error(y_true, data['LR_prediction']),
            'r2': r2_score(y_true, data['LR_prediction'])
        },
        'weighted_average': {
            'mse': mean_squared_error(y_true, data['weighted_average']),
            'mae': mean_absolute_error(y_true, data['weighted_average']),
            'r2': r2_score(y_true, data['weighted_average'])
        }
    }
    return metrics

#### Function to perform the entire stacking and metrics calculation of one reservoir

In [ ]:
def full_workflow_for_reservoir(reservoir_id, years_training, df=df):
    
    train_sarima_prophet, train, test, capacity, years = prepare_data(df, reservoir_id)

    train, test = cv_sarima(train_sarima_prophet, train, test, years, capacity)

    train, test = cv_prophet(train_sarima_prophet, train, test, years, capacity)

    # Prepare data for XGBoost
    train_series_xgboost = train.copy()
    train = train[-(52*years_training):]
    train.loc[:, 'storage'] = train['storage'].astype(float)
    train.loc[:, 'sarima_prediction'] = train['sarima_prediction'].astype(float)
    train.loc[:, 'prophet_prediction'] = train['prophet_prediction'].astype(float)

    train['xgboost_prediction'] = np.nan

    test.loc[:, 'storage'] = test['storage'].astype(float)
    test.loc[:, 'sarima_prediction'] = test['sarima_prediction'].astype(float)
    test.loc[:, 'prophet_prediction'] = test['prophet_prediction'].astype(float)

    test['xgboost_prediction'] = np.nan

    years_training_xgboost = (len(train_series_xgboost) // 52) - years_training

    train, test = cv_xgboost(train, test, train_series_xgboost, years_training, years_training_xgboost, capacity)

    # Workflow for linear regression
    X_train = train[['sarima_prediction', 'prophet_prediction', 'xgboost_prediction']].copy()
    y_train = train['storage'].copy()

    X_test = test[['sarima_prediction', 'prophet_prediction', 'xgboost_prediction']].copy()

    # Train the linear regression stacking model
    stacking_model = LinearRegression()
    stacking_model.fit(X_train, y_train)

    LR_prediction_list = stacking_model.predict(X_test)

    data_for_metrics = test.copy()
    data_for_metrics['LR_prediction'] = LR_prediction_list
    data_for_metrics['weighted_average'] = data_for_metrics['prophet_prediction']*0.15 + data_for_metrics['sarima_prediction']*0.35 + data_for_metrics['xgboost_prediction']*0.50

    plot_one_reservoir(train, data_for_metrics, reservoir_id)

    reservoir_metrics = calculate_reservoir_metrics(data_for_metrics)

    return reservoir_metrics

# Comparison against 10 random samples

Generating the random reservoirs

In [ ]:
numbers = random.sample(range(0, len(df['id'].unique())), 10)

In [ ]:
df['id'].unique()[numbers]

#### Cell that compares different models for these random reservoirs

In [ ]:
metrics = {
    'prophet': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'sarima': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'xgboost': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'linear_regression': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'weighted_average': {
        'mse': [],
        'mae': [],
        'r2': []
    }
}

averaged_metrics = {
    'prophet': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'sarima': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'xgboost': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'linear_regression': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'weighted_average': {
        'mse': [],
        'mae': [],
        'r2': []
    }
}

for reservoir in df['id'].unique()[numbers]:
    try:
        reservoir_metrics = full_workflow_for_reservoir(reservoir, 4)
    except Exception as e:
        print(f"Skipped {reservoir} due to error: {e}")
        continue

    for model in metrics:
        for metric in metrics[model]:
            metrics[model][metric].append(reservoir_metrics[model][metric])

# Calculate and print the average metrics for each model
for model in metrics:
    for metric in metrics[model]:
        averaged_metrics[model][metric] = np.mean(metrics[model][metric])
    print(f"\n Average metrics for {model}: MSE={averaged_metrics[model]['mse']}, MAE={averaged_metrics[model]['mae']}, R2={averaged_metrics[model]['r2']} \n")


Here it can be seen that:
- Linear regression is the best model
- The weighted average is not that far, and doesn't consider linear regression in that average
- It suggests that a weighted average with linear regression may outperform every model

# Testing on the 10 reservoirs with most capacity

In [ ]:
df.sort_values('capacity', ascending=False)['id'].unique()[:10]

### Will change the name of weighted average to full weighted average, as it is going to be tried a variation that embraces all the models (before, Linear Regression wasn't taken into account) because it is thought that it may outperform the rest of models

Defining those functions that change due to that name modification:

In [ ]:
def calculate_reservoir_metrics(data):
    y_true = data['storage']
    metrics = {
        'prophet': {
            'mse': mean_squared_error(y_true, data['prophet_prediction']),
            'mae': mean_absolute_error(y_true, data['prophet_prediction']),
            'r2': r2_score(y_true, data['prophet_prediction'])
        },
        'sarima': {
            'mse': mean_squared_error(y_true, data['sarima_prediction']),
            'mae': mean_absolute_error(y_true, data['sarima_prediction']),
            'r2': r2_score(y_true, data['sarima_prediction'])
        },
        'xgboost': {
            'mse': mean_squared_error(y_true, data['xgboost_prediction']),
            'mae': mean_absolute_error(y_true, data['xgboost_prediction']),
            'r2': r2_score(y_true, data['xgboost_prediction'])
        },
        'linear_regression': {
            'mse': mean_squared_error(y_true, data['LR_prediction']),
            'mae': mean_absolute_error(y_true, data['LR_prediction']),
            'r2': r2_score(y_true, data['LR_prediction'])
        },
        'full_weighted_average': {
            'mse': mean_squared_error(y_true, data['full_weighted_average']),
            'mae': mean_absolute_error(y_true, data['full_weighted_average']),
            'r2': r2_score(y_true, data['full_weighted_average'])
        }
    }
    return metrics

In [ ]:
def plot_one_reservoir(train, data_for_metrics, reservoir_id):
# Create comprehensive plot
    plt.figure(figsize=(15, 8))

    # Plot training data and predictions
    plt.plot(train.index, train['storage'], label='Train Actual', color='blue', linewidth=2)
    plt.plot(train.index, train['sarima_prediction'], label='SARIMA Train', color='red', linestyle='--', alpha=0.7)
    plt.plot(train.index, train['prophet_prediction'], label='Prophet Train', color='green', linestyle='--', alpha=0.7)
    plt.plot(train.index, train['xgboost_prediction'], label='XGBoost Train', color='purple', linestyle='--', alpha=0.7)

    # Plot test data and predictions
    plt.plot(data_for_metrics.index, data_for_metrics['storage'], label='Test Actual', color='black', linewidth=3)
    plt.plot(data_for_metrics.index, data_for_metrics['sarima_prediction'], label='SARIMA Test', color='red', alpha=0.8)
    plt.plot(data_for_metrics.index, data_for_metrics['prophet_prediction'], label='Prophet Test', color='green', alpha=0.8)
    plt.plot(data_for_metrics.index, data_for_metrics['xgboost_prediction'], label='XGBoost Test', color='purple', alpha=0.8)
    plt.plot(data_for_metrics.index, data_for_metrics['LR_prediction'], label='Stacking Test', color='orange', linewidth=2)
    plt.plot(data_for_metrics.index, data_for_metrics['full_weighted_average'], label='Full Weighted Average Test', color='cyan', linestyle='--', alpha=0.8)

    # Add train/test split line
    plt.axvline(data_for_metrics.index[0], color='gray', linestyle='--', alpha=0.8, label='Train/Test Split')

    # Formatting
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f'Stacking Ensemble Forecast for Reservoir {reservoir_id}')
    plt.xlabel('Date')
    plt.ylabel('Storage')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
def full_workflow_for_reservoir_with_linear_regression_ponderation(reservoir_id, years_training, df=df):
    
    train_sarima_prophet, train, test, capacity, years = prepare_data(df, reservoir_id)

    train, test = cv_sarima(train_sarima_prophet, train, test, years, capacity)

    train, test = cv_prophet(train_sarima_prophet, train, test, years, capacity)

    # Prepare data for XGBoost
    train_series_xgboost = train.copy()
    train = train[-(52*years_training):]
    train.loc[:, 'storage'] = train['storage'].astype(float)
    train.loc[:, 'sarima_prediction'] = train['sarima_prediction'].astype(float)
    train.loc[:, 'prophet_prediction'] = train['prophet_prediction'].astype(float)

    train['xgboost_prediction'] = np.nan

    test.loc[:, 'storage'] = test['storage'].astype(float)
    test.loc[:, 'sarima_prediction'] = test['sarima_prediction'].astype(float)
    test.loc[:, 'prophet_prediction'] = test['prophet_prediction'].astype(float)

    test['xgboost_prediction'] = np.nan

    years_training_xgboost = (len(train_series_xgboost) // 52) - years_training

    train, test = cv_xgboost(train, test, train_series_xgboost, years_training, years_training_xgboost, capacity)

    # Workflow for linear regression
    X_train = train[['sarima_prediction', 'prophet_prediction', 'xgboost_prediction']].copy()
    y_train = train['storage'].copy()

    X_test = test[['sarima_prediction', 'prophet_prediction', 'xgboost_prediction']].copy()

    # Train the linear regression stacking model
    stacking_model = LinearRegression()
    stacking_model.fit(X_train, y_train)

    LR_prediction_list = stacking_model.predict(X_test)

    data_for_metrics = test.copy()
    data_for_metrics['LR_prediction'] = LR_prediction_list
    data_for_metrics['full_weighted_average'] = data_for_metrics['prophet_prediction']*0.15 + data_for_metrics['sarima_prediction']*0.25 + data_for_metrics['xgboost_prediction']*0.25 + data_for_metrics['LR_prediction']*0.35

    plot_one_reservoir(train, data_for_metrics, reservoir_id)

    reservoir_metrics = calculate_reservoir_metrics(data_for_metrics)

    return reservoir_metrics

#### Cell that makes the comparison:

In [ ]:
metrics = {
    'prophet': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'sarima': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'xgboost': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'linear_regression': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'full_weighted_average': {
        'mse': [],
        'mae': [],
        'r2': []
    }
}

averaged_metrics = {
    'prophet': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'sarima': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'xgboost': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'linear_regression': {
        'mse': [],
        'mae': [],
        'r2': []
    },
    'full_weighted_average': {
        'mse': [],
        'mae': [],
        'r2': []
    }
}

for reservoir in df.sort_values('capacity', ascending=False)['id'].unique()[:10]:
    try:
        reservoir_metrics = full_workflow_for_reservoir_with_linear_regression_ponderation(reservoir, 4)
    except Exception as e:
        print(f"Skipped {reservoir} due to error: {e}")
        continue

    for model in metrics:
        for metric in metrics[model]:
            metrics[model][metric].append(reservoir_metrics[model][metric])

# Calculate and print the average metrics for each model
for model in metrics:
    for metric in metrics[model]:
        averaged_metrics[model][metric] = np.mean(metrics[model][metric])
    print(f"\n Average metrics for {model}: MSE={averaged_metrics[model]['mse']}, MAE={averaged_metrics[model]['mae']}, R2={averaged_metrics[model]['r2']} \n")


As full weighted average has the least MSE, this is the model that will be used for the final predictions